# 03_pc_<agreement>_<source>_to_<target> — deterministic pipeline enforcement

Run-all-safe pipeline notebook for parameter-driven contract execution using approved metadata and deterministic DQ enforcement.


## 1. Runtime setup and pipeline parameters


In [ ]:
%run 00_env_config

from pyspark.sql import functions as F

from fabricops_kit import (
    load_agreements,
    select_agreement,
    get_selected_agreement,
    register_current_notebook,
    load_fabric_config,
    setup_fabricops_notebook,
    get_path,
    lakehouse_table_read,
    warehouse_read,
    lakehouse_table_write,
    warehouse_write,
    profile_dataframe,
    check_schema_drift,
    check_partition_drift,
    check_profile_drift,
    summarize_drift_results,
    standardize_output_columns,
    enforce_dq_rules,
    assert_dq_passed,
    build_lineage_records,
    build_lineage_handover_markdown,
    build_run_summary,
    render_run_summary_markdown,
)


In [ ]:
USE_SAMPLE_DATA = True

ENV_NAME = "dev"
SOURCE_KIND = "lakehouse"
TARGET_KIND = "lakehouse"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else "TODO_target_table"
DQ_TABLE_NAME = TARGET_TABLE
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "target_dataset"
PIPELINE_NAME = f"{SOURCE_TABLE}_to_{TARGET_TABLE}"
WRITE_MODE = "overwrite"
REQUIRED_SOURCE_COLUMNS = ["customer_id", "event_ts", "status", "amount"]
BUSINESS_KEYS = ["customer_id"]
RUN_ID = f"{PIPELINE_NAME}_{ENV_NAME}"
DRIFT_MODE = "warn"  # one of: warn, fail
DQ_PUBLISH_MODE = "split_valid_quarantine"  # same_table_with_flags | split_valid_quarantine | fail_on_invalid

CONFIG = load_fabric_config(CONFIG)
if not USE_SAMPLE_DATA:
    setup_fabricops_notebook(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])

source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)

runtime_context = {
    "run_id": RUN_ID,
    "environment": ENV_NAME,
    "dataset_name": DATASET_NAME,
    "source_table": SOURCE_TABLE,
    "target_table": TARGET_TABLE,
    "pipeline_name": PIPELINE_NAME,
    "started_at_utc": RUN_ID,
}


## 2. Load agreement context and source data


In [ ]:
agreements = load_agreements(spark)
select_agreement(agreements)

selected_agreement = get_selected_agreement()
agreement_id = selected_agreement["agreement_id"]

register_current_notebook(
    spark,
    metadata_path=metadata_path,
    agreement_id=agreement_id,
    notebook_type="03_pc",
    environment_name=ENV_NAME,
    dataset_name=DATASET_NAME,
    table_name=TARGET_TABLE,
    pipeline_name=PIPELINE_NAME,
)

if SOURCE_KIND == "lakehouse":
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(ENV_NAME, SOURCE_LAYER, "dbo", SOURCE_TABLE, config=CONFIG)


## 3. Source evidence and guardrails


In [ ]:
missing = sorted(set(REQUIRED_SOURCE_COLUMNS) - set(df_source.columns))
if missing:
    raise ValueError(f"Fail fast: missing required source columns: {missing}")

source_profile = profile_dataframe(df_source, SOURCE_TABLE)
current_profile_metrics = {"row_count": int(df_source.count()), "columns": []}
schema_drift_result = check_schema_drift(df_source, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE, baseline_snapshot=None)
partition_drift_result = check_partition_drift(
    df_source,
    dataset_name=DATASET_NAME,
    table_name=SOURCE_TABLE,
    partition_column="event_ts",
    business_keys=BUSINESS_KEYS,
    baseline_snapshot=None,
    run_id=RUN_ID,
)
profile_drift_result = check_profile_drift(current_profile_metrics, baseline_profile=None)
drift_summary = summarize_drift_results(
    schema_drift_result=schema_drift_result,
    partition_drift_result=partition_drift_result,
    profile_drift_result=profile_drift_result,
)

if DRIFT_MODE == "fail" and drift_summary.get("status") not in {"ok", "pass", "passed", "no_baseline"}:
    raise ValueError(f"Drift checks failed in fail mode: {drift_summary}")

display(source_profile)
display(drift_summary)


## 4. Deterministic transformation and technical columns


In [ ]:
df_transformed = (
    df_source.withColumn("status", F.trim(F.lower(F.col("status"))))
    .withColumn("email", F.trim(F.lower(F.col("email"))))
    .withColumn("amount", F.col("amount").cast("double"))
)

df_standard = standardize_output_columns(
    df_transformed,
    run_id=RUN_ID,
    pipeline_name=PIPELINE_NAME,
    environment=ENV_NAME,
    source_table=SOURCE_TABLE,
    business_keys=BUSINESS_KEYS,
    bucket_column=BUSINESS_KEYS[0] if BUSINESS_KEYS else None,
)


## 5. DQ enforcement and evidence materialization


In [ ]:
metadata_dq_rules = spark.table("METADATA_DQ_RULES")
dq = enforce_dq_rules(
    df_standard,
    table_name=DQ_TABLE_NAME,
    metadata_df=metadata_dq_rules,
    dq_run_id=RUN_ID,
)

df_valid = dq.valid_rows
df_quarantine = dq.quarantine_rows
dq_failure_evidence = dq.failure_rows
dq_rule_results = dq.rule_results
df_dq_full = dq.output_df if hasattr(dq, "output_df") else df_standard

failure_table = f"{TARGET_TABLE}_DQ_FAILURES"
rules_table = f"{TARGET_TABLE}_DQ_RULE_RESULTS"
lakehouse_table_write(dq_failure_evidence, metadata_path, failure_table, mode=WRITE_MODE)
lakehouse_table_write(dq_rule_results, metadata_path, rules_table, mode=WRITE_MODE)

invalid_row_count = df_quarantine.count()
if DQ_PUBLISH_MODE == "same_table_with_flags":
    publish_df = df_dq_full
elif DQ_PUBLISH_MODE == "split_valid_quarantine":
    publish_df = df_valid
elif DQ_PUBLISH_MODE == "fail_on_invalid":
    if invalid_row_count > 0:
        raise ValueError(f"DQ_PUBLISH_MODE=fail_on_invalid blocked publish with {invalid_row_count} invalid rows.")
    publish_df = df_valid
else:
    raise ValueError(f"Unsupported DQ_PUBLISH_MODE: {DQ_PUBLISH_MODE}")

assert_dq_passed(dq_rule_results)


## 6. Publish strategy


In [ ]:
quarantine_table = f"{TARGET_TABLE}_QUARANTINE"

if DQ_PUBLISH_MODE == "same_table_with_flags":
    # Keep all rows in target; use mainly when DQ rules are warning/advisory.
    if TARGET_KIND == "lakehouse":
        lakehouse_table_write(publish_df, target_path, TARGET_TABLE, mode=WRITE_MODE)
    else:
        warehouse_write(publish_df, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)
elif DQ_PUBLISH_MODE == "split_valid_quarantine":
    if TARGET_KIND == "lakehouse":
        lakehouse_table_write(df_valid, target_path, TARGET_TABLE, mode=WRITE_MODE)
        lakehouse_table_write(df_quarantine, target_path, quarantine_table, mode=WRITE_MODE)
    else:
        warehouse_write(df_valid, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)
        warehouse_write(df_quarantine, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=quarantine_table, mode=WRITE_MODE, config=CONFIG)
elif DQ_PUBLISH_MODE == "fail_on_invalid":
    if TARGET_KIND == "lakehouse":
        lakehouse_table_write(publish_df, target_path, TARGET_TABLE, mode=WRITE_MODE)
    else:
        warehouse_write(publish_df, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)


## 7. Post-write profile, lineage, and run summary


In [ ]:
if TARGET_KIND == "lakehouse":
    df_published = lakehouse_table_read(target_path, TARGET_TABLE)
else:
    df_published = warehouse_read(ENV_NAME, TARGET_LAYER, "dbo", TARGET_TABLE, config=CONFIG)

published_profile = profile_dataframe(df_published, TARGET_TABLE)
published_profile = (
    published_profile
    .withColumn("DATASET_NAME", F.lit(DATASET_NAME))
    .withColumn("RUN_ID", F.lit(RUN_ID))
    .withColumn("PIPELINE_NAME", F.lit(PIPELINE_NAME))
)
lakehouse_table_write(published_profile, metadata_path, f"{TARGET_TABLE}_PUBLISHED_PROFILE", mode=WRITE_MODE)

lineage_records = build_lineage_records(
    dataset_name=DATASET_NAME,
    run_id=RUN_ID,
    source_tables=[SOURCE_TABLE],
    target_table=TARGET_TABLE,
    transformation_steps=[
        {"step": 1, "source": SOURCE_TABLE, "target": TARGET_TABLE, "operation": "deterministic transform + technical columns"},
        {"step": 2, "source": TARGET_TABLE, "target": quarantine_table, "operation": f"publish strategy: {DQ_PUBLISH_MODE}"},
    ],
)
lineage_markdown = build_lineage_handover_markdown(lineage_records, dataset_name=DATASET_NAME, run_id=RUN_ID)

run_summary = build_run_summary(
    runtime_context=runtime_context,
    source_profile={"row_count": int(df_source.count()), "columns": []},
    output_profile={"row_count": int(df_published.count()), "columns": []},
    schema_drift_result=schema_drift_result,
    incremental_safety_result=partition_drift_result,
    quality_result={"status": "passed", "rule_count": len(dq.rules)},
    lineage_summary={"record_count": len(lineage_records), "markdown": lineage_markdown},
)

run_summary_markdown = render_run_summary_markdown(run_summary)
print(lineage_markdown)
print(run_summary_markdown)

handover_summary = {
    "run_id": RUN_ID,
    "pipeline_name": PIPELINE_NAME,
    "dq_publish_mode": DQ_PUBLISH_MODE,
    "source_count": df_source.count(),
    "published_count": df_published.count(),
    "quarantine_count": df_quarantine.count(),
    "target_table": TARGET_TABLE,
    "quarantine_table": quarantine_table,
    "dq_rule_count": len(dq.rules),
    "drift_mode": DRIFT_MODE,
    "drift_status": drift_summary.get("status"),
}
display(handover_summary)
